# TBM-X — 09B MODEL INTEGRATION & VALIDATION

## Physics Model Integration

Bu notebook, TBM-X Physics Model'in ana zincirini bir araya getirir:

**Weight → Geometry → Aerodynamics → Propulsion → Performance**

Amaç, önceki notebook'larda geliştirilen modellerin tek bir preliminary design state üzerinden birlikte çalışmasını sağlamaktır.

> Bu notebook bir model entegrasyon kontrol noktasıdır. Henüz nihai tasarım veya sertifikasyon hesabı değildir.

### Ana prensip

**Benchmark'a zorla uydurma değil, benchmark ile doğrulama.**


In [1]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("✓ Libraries imported..")


✓ Libraries imported.


In [2]:
# ============================================================
# CELL 3 — WORKING DIRECTORY
# ============================================================

PROJECT_ROOT = Path.cwd()

print("Project root:")
print(PROJECT_ROOT)


Project root:
C:\Users\musta\Documents\RAYMER\TBM-X_v0.2_data_engineering


In [3]:
# ============================================================
# CELL 4 — OUTPUT PATHS
# ============================================================

integration_output_dir = PROJECT_ROOT / "outputs" / "integration"
integration_output_dir.mkdir(parents=True, exist_ok=True)

print("Integration output directory:")
print(integration_output_dir)


Integration output directory:
C:\Users\musta\Documents\RAYMER\TBM-X_v0.2_data_engineering\outputs\integration


## 5 — TBM-X Preliminary Requirements

Şimdilik 05–09 notebook'larında kullandığımız preliminary değerleri koruyoruz. Daha sonra requirements modülü ile merkezi hale getirilebilir.


In [4]:
# ============================================================
# CELL 5 — REQUIREMENTS
# ============================================================

requirements = {
    "aircraft_category": "Single Engine Turboprop",
    "crew": 1,
    "passengers": 5,
    "range_km": 3000.0,
    "cruise_speed_kt": 300.0,
    "cruise_altitude_ft": 25000.0,
    "service_ceiling_ft": 31000.0,
    "initial_mtow_kg": 3400.0,
    "stall_speed_kt": 65.0,
}

requirements


{'aircraft_category': 'Single Engine Turboprop',
 'crew': 1,
 'passengers': 5,
 'range_km': 3000.0,
 'cruise_speed_kt': 300.0,
 'cruise_altitude_ft': 25000.0,
 'service_ceiling_ft': 31000.0,
 'initial_mtow_kg': 3400.0,
 'stall_speed_kt': 65.0}

In [5]:
# ============================================================
# CELL 6 — DESIGN STATE
# ============================================================

design_state = {
    "MTOW_kg": 3400.0,
    "Wing_Area_m2": 18.0,
    "Aspect_Ratio": 9.15,
    "Engine_Power_kW": 650.0,
    "CD0": 0.025,
    "Oswald_Efficiency": 0.80,
    "Propeller_Efficiency": 0.85,
    "BSFC_kg_kWh": 0.30,
    "CLmax": 1.80,
}

design_state


{'MTOW_kg': 3400.0,
 'Wing_Area_m2': 18.0,
 'Aspect_Ratio': 9.15,
 'Engine_Power_kW': 650.0,
 'CD0': 0.025,
 'Oswald_Efficiency': 0.8,
 'Propeller_Efficiency': 0.85,
 'BSFC_kg_kWh': 0.3,
 'CLmax': 1.8}

In [6]:
# ============================================================
# CELL 7 — DESIGN STATE INSPECTION
# ============================================================

design_state_df = pd.DataFrame(
    design_state.items(),
    columns=["Parameter", "Value"]
)

design_state_df


,Parameter,Value
0,MTOW_kg,"3,400.0000"
1,Wing_Area_m2,18.0000
2,Aspect_Ratio,9.1500
3,Engine_Power_kW,650.0000
4,CD0,0.0250
5,Oswald_Efficiency,0.8000
6,Propeller_Efficiency,0.8500
7,BSFC_kg_kWh,0.3000
8,CLmax,1.8000


## 8 — Weight → Geometry

İlk entegrasyon zinciri:

**MTOW → Wing Area → Aspect Ratio → Span → Mean Chord**


In [7]:
# ============================================================
# CELL 8 — WEIGHT → GEOMETRY
# ============================================================

MTOW_kg = design_state["MTOW_kg"]
S = design_state["Wing_Area_m2"]
AR = design_state["Aspect_Ratio"]

g = 9.80665
W_N = MTOW_kg * g

print(f"MTOW        : {MTOW_kg:.1f} kg")
print(f"Wing area   : {S:.2f} m²")
print(f"Aspect ratio: {AR:.2f}")


MTOW        : 3400.0 kg
Wing area   : 18.00 m²
Aspect ratio: 9.15


In [8]:
# ============================================================
# CELL 9 — GEOMETRY CALCULATION
# ============================================================

span = math.sqrt(AR * S)
mean_chord = S / span
wing_loading = MTOW_kg / S

geometry = {
    "MTOW_kg": MTOW_kg,
    "Wing_Area_m2": S,
    "Aspect_Ratio": AR,
    "Span_m": span,
    "Mean_Chord_m": mean_chord,
    "Wing_Loading_kg_m2": wing_loading,
}

geometry


{'MTOW_kg': 3400.0,
 'Wing_Area_m2': 18.0,
 'Aspect_Ratio': 9.15,
 'Span_m': 12.833549781724463,
 'Mean_Chord_m': 1.4025737466365533,
 'Wing_Loading_kg_m2': 188.88888888888889}

In [9]:
# ============================================================
# CELL 10 — GEOMETRY CLOSURE
# ============================================================

AR_check = span**2 / S

print(f"Calculated span       : {span:.4f} m")
print(f"Calculated mean chord : {mean_chord:.4f} m")
print(f"Calculated AR         : {AR_check:.6f}")

if abs(AR_check - AR) < 1e-10:
    print("✓ Geometry closure satisfied.")
else:
    print("⚠ Geometry closure problem.")


Calculated span       : 12.8335 m
Calculated mean chord : 1.4026 m
Calculated AR         : 9.150000
✓ Geometry closure satisfied.


## 11 — Geometry → Aerodynamics

Aerodinamik model:

**CL → CD → L/D → Drag → Required Shaft Power**

Induced drag coefficient:

**k = 1 / (π e AR)**


In [10]:
# ============================================================
# CELL 11 — GEOMETRY → AERODYNAMICS
# ============================================================

e = design_state["Oswald_Efficiency"]
CD0 = design_state["CD0"]

k = 1.0 / (math.pi * e * AR)

aero_parameters = {
    "CD0": CD0,
    "Oswald_Efficiency": e,
    "Aspect_Ratio": AR,
    "k": k,
}

aero_parameters


{'CD0': 0.025,
 'Oswald_Efficiency': 0.8,
 'Aspect_Ratio': 9.15,
 'k': 0.043484957128933154}

In [11]:
# ============================================================
# CELL 12 — CRUISE CONDITION
# ============================================================

cruise_speed_kt = requirements["cruise_speed_kt"]
cruise_speed_ms = cruise_speed_kt * 0.514444

cruise_altitude_ft = requirements["cruise_altitude_ft"]
cruise_altitude_m = cruise_altitude_ft * 0.3048

print(f"Cruise speed    : {cruise_speed_kt:.1f} kt")
print(f"Cruise speed    : {cruise_speed_ms:.2f} m/s")
print(f"Cruise altitude : {cruise_altitude_ft:.0f} ft")
print(f"Cruise altitude : {cruise_altitude_m:.1f} m")


Cruise speed    : 300.0 kt
Cruise speed    : 154.33 m/s
Cruise altitude : 25000 ft
Cruise altitude : 7620.0 m


In [12]:
# ============================================================
# CELL 13 — ISA ATMOSPHERE
# ============================================================

def isa_atmosphere(altitude_m):
    """Simplified ISA atmosphere in the troposphere."""

    T0 = 288.15
    P0 = 101325.0
    lapse_rate = 0.0065
    R = 287.05
    g = 9.80665

    T = T0 - lapse_rate * altitude_m
    P = P0 * (T / T0) ** (g / (R * lapse_rate))
    rho = P / (R * T)

    return T, P, rho


T, P, rho = isa_atmosphere(cruise_altitude_m)

print(f"Temperature : {T:.2f} K")
print(f"Pressure    : {P:.1f} Pa")
print(f"Density     : {rho:.4f} kg/m³")


Temperature : 238.62 K
Pressure    : 37600.5 Pa
Density     : 0.5489 kg/m³


In [13]:
# ============================================================
# CELL 14 — AERODYNAMIC CALCULATION
# ============================================================

q = 0.5 * rho * cruise_speed_ms**2

CL = W_N / (q * S)
CDi = k * CL**2
CD = CD0 + CDi
L_D = CL / CD
Drag_N = q * S * CD

print(f"Dynamic pressure : {q:.2f} Pa")
print(f"CL               : {CL:.4f}")
print(f"CDi              : {CDi:.5f}")
print(f"CD               : {CD:.5f}")
print(f"L/D              : {L_D:.2f}")
print(f"Drag             : {Drag_N:,.1f} N")


Dynamic pressure : 6537.60 Pa
CL               : 0.2833
CDi              : 0.00349
CD               : 0.02849
L/D              : 9.94
Drag             : 3,352.7 N


In [14]:
# ============================================================
# CELL 15 — STORE AERODYNAMIC RESULTS
# ============================================================

aero_results = {
    "Density_kg_m3": rho,
    "Dynamic_Pressure_Pa": q,
    "CL": CL,
    "CDi": CDi,
    "CD": CD,
    "Lift_to_Drag": L_D,
    "Drag_N": Drag_N,
}

aero_results


{'Density_kg_m3': 0.5489457550758864,
 'Dynamic_Pressure_Pa': 6537.5971800246025,
 'CL': 0.2833406787255209,
 'CDi': 0.0034910567287220542,
 'CD': 0.028491056728722056,
 'Lift_to_Drag': 9.944898900147953,
 'Drag_N': 3352.7349382610573}

## 16 — Aerodynamics → Propulsion

Gerekli shaft power:

**P_shaft = D × V / η_p**

Burada propeller efficiency yalnızca bir kez uygulanır.


In [15]:
# ============================================================
# CELL 16 — REQUIRED SHAFT POWER
# ============================================================

eta_p = design_state["Propeller_Efficiency"]

propulsive_power_kW = Drag_N * cruise_speed_ms / 1000.0
required_shaft_power_kW = propulsive_power_kW / eta_p

print(f"Propulsive power     : {propulsive_power_kW:.2f} kW")
print(f"Required shaft power : {required_shaft_power_kW:.2f} kW")


Propulsive power     : 517.44 kW
Required shaft power : 608.75 kW


In [16]:
# ============================================================
# CELL 17 — PROPULSION MODEL
# ============================================================

engine_power_kW = design_state["Engine_Power_kW"]

power_margin_kW = engine_power_kW - required_shaft_power_kW
power_margin_fraction = power_margin_kW / engine_power_kW

BSFC = design_state["BSFC_kg_kWh"]
fuel_flow_kg_h = BSFC * required_shaft_power_kW

propulsion_results = {
    "Engine_Power_kW": engine_power_kW,
    "Required_Shaft_Power_kW": required_shaft_power_kW,
    "Power_Margin_kW": power_margin_kW,
    "Power_Margin_fraction": power_margin_fraction,
    "BSFC_kg_kWh": BSFC,
    "Fuel_Flow_kg_h": fuel_flow_kg_h,
}

propulsion_results


{'Engine_Power_kW': 650.0,
 'Required_Shaft_Power_kW': 608.7509550278016,
 'Power_Margin_kW': 41.24904497219836,
 'Power_Margin_fraction': 0.06346006918799749,
 'BSFC_kg_kWh': 0.3,
 'Fuel_Flow_kg_h': 182.62528650834048}

In [17]:
# ============================================================
# CELL 18 — POWER BALANCE
# ============================================================

print(f"Engine available power : {engine_power_kW:.2f} kW")
print(f"Required shaft power   : {required_shaft_power_kW:.2f} kW")
print(f"Power margin           : {power_margin_kW:.2f} kW")

if power_margin_kW >= 0:
    print("✓ Preliminary cruise power balance satisfied.")
else:
    print("⚠ Insufficient cruise shaft power.")


Engine available power : 650.00 kW
Required shaft power   : 608.75 kW
Power margin           : 41.25 kW
✓ Preliminary cruise power balance satisfied.


## 19 — Preliminary Rate of Climb

Bu aşamada ROC için preliminary excess-power yaklaşımı kullanıyoruz.

**ROC ≈ Excess Power / Weight**

Bu, tam climb-performance modeli değildir; entegrasyon kontrolüdür.


In [18]:
# ============================================================
# CELL 19 — PRELIMINARY RATE OF CLIMB
# ============================================================

excess_power_kW = engine_power_kW - required_shaft_power_kW

ROC_ms = excess_power_kW * 1000.0 / W_N
ROC_ft_min = ROC_ms * 196.8504

print(f"Excess power : {excess_power_kW:.2f} kW")
print(f"ROC estimate : {ROC_ms:.2f} m/s")
print(f"ROC estimate : {ROC_ft_min:.0f} ft/min")


Excess power : 41.25 kW
ROC estimate : 1.24 m/s
ROC estimate : 244 ft/min


In [19]:
# ============================================================
# CELL 20 — INTEGRATED CHAIN SUMMARY
# ============================================================

integrated_summary = {
    "MTOW_kg": MTOW_kg,
    "Wing_Area_m2": S,
    "Aspect_Ratio": AR,
    "Span_m": span,
    "Mean_Chord_m": mean_chord,
    "Wing_Loading_kg_m2": wing_loading,
    "Cruise_Speed_kt": cruise_speed_kt,
    "Cruise_Altitude_ft": cruise_altitude_ft,
    "Density_kg_m3": rho,
    "CL": CL,
    "CD": CD,
    "Lift_to_Drag": L_D,
    "Drag_N": Drag_N,
    "Required_Shaft_Power_kW": required_shaft_power_kW,
    "Engine_Power_kW": engine_power_kW,
    "Power_Margin_kW": power_margin_kW,
    "Fuel_Flow_kg_h": fuel_flow_kg_h,
    "ROC_ft_min": ROC_ft_min,
}

integrated_summary_df = pd.DataFrame([integrated_summary])
integrated_summary_df.T


,0
MTOW_kg,"3,400.0000"
Wing_Area_m2,18.0000
Aspect_Ratio,9.1500
Span_m,12.8335
Mean_Chord_m,1.4026
Wing_Loading_kg_m2,188.8889
Cruise_Speed_kt,300.0000
Cruise_Altitude_ft,"25,000.0000"
Density_kg_m3,0.5489
CL,0.2833


## 21 — Integration Checks

Burada amaç sadece sonuç üretmek değil, model zincirinin birbirine tutarlı bağlandığını kontrol etmektir.


In [20]:
# ============================================================
# CELL 21 — INTEGRATION CHECKS
# ============================================================

checks = {
    "MTOW_positive": MTOW_kg > 0,
    "Wing_area_positive": S > 0,
    "Aspect_ratio_positive": AR > 0,
    "Density_positive": rho > 0,
    "CL_positive": CL > 0,
    "CD_positive": CD > 0,
    "L_D_positive": L_D > 0,
    "Required_power_positive": required_shaft_power_kW > 0,
    "Fuel_flow_positive": fuel_flow_kg_h > 0,
}

checks_df = pd.DataFrame(
    checks.items(),
    columns=["Check", "Status"]
)

checks_df


,Check,Status
0,MTOW_positive,True
1,Wing_area_positive,True
2,Aspect_ratio_positive,True
3,Density_positive,True
4,CL_positive,True
5,CD_positive,True
6,L_D_positive,True
7,Required_power_positive,True
8,Fuel_flow_positive,True


In [21]:
# ============================================================
# CELL 22 — FINAL INTEGRATION STATUS
# ============================================================

all_checks_pass = all(checks.values())

if all_checks_pass:
    integration_status = "PASS"
    print("✓ TBM-X preliminary model integration: PASS")
else:
    integration_status = "REVIEW"
    print("⚠ TBM-X preliminary model integration requires review.")


✓ TBM-X preliminary model integration: PASS


In [22]:
# ============================================================
# CELL 23 — FINAL DESIGN VECTOR
# ============================================================

design_vector = {
    **geometry,
    **aero_parameters,
    **aero_results,
    **propulsion_results,
    "Cruise_Speed_kt": cruise_speed_kt,
    "Cruise_Altitude_ft": cruise_altitude_ft,
    "ROC_ft_min": ROC_ft_min,
    "Integration_Status": integration_status,
}

design_vector_df = pd.DataFrame([design_vector])
design_vector_df.T


,0
MTOW_kg,"3,400.0000"
Wing_Area_m2,18.0000
Aspect_Ratio,9.1500
Span_m,12.8335
Mean_Chord_m,1.4026
Wing_Loading_kg_m2,188.8889
CD0,0.0250
Oswald_Efficiency,0.8000
k,0.0435
Density_kg_m3,0.5489


In [23]:
# ============================================================
# CELL 24 — SAVE INTEGRATED DESIGN VECTOR
# ============================================================

design_vector_path = (
    integration_output_dir
    / "TBM_X_Integrated_Design_Vector_v0_1.csv"
)

design_vector_df.to_csv(
    design_vector_path,
    index=False
)

print("✓ Integrated design vector saved:")
print(design_vector_path)


✓ Integrated design vector saved:
C:\Users\musta\Documents\RAYMER\TBM-X_v0.2_data_engineering\outputs\integration\TBM_X_Integrated_Design_Vector_v0_1.csv


# 25 — CONCLUSION

09B completes the first integrated preliminary physics chain:

**Weight → Geometry → Aerodynamics → Propulsion → Performance**

The result is stored as a compact design vector.

### Important limitations

- MTOW is still a preliminary working value.
- Wing area and aspect ratio are preliminary.
- CD0, Oswald efficiency, propeller efficiency and BSFC contain engineering assumptions.
- ROC is a preliminary excess-power estimate.
- This notebook is an integration checkpoint, not a certification model.

The next physics-model step is **Stability & Control**, followed by mission integration and benchmark validation.
